# Colab — Calcul des VaR rolling du papier

Ce notebook est **autonome** : il recrée les modules nécessaires (`msm.py`, `copulas.py`, `var_v2.py`) dans l'environnement Colab, puis charge uniquement le CSV des rendements depuis Google Drive.

Objectif : lancer les VaR **une par une**, sauvegarder chaque résultat dans Drive, puis concaténer les fichiers obtenus.

Convention VaR utilisée : VaR signée, généralement négative.

\[
P(r_{p,t} \leq VaR_t(lpha) \mid \Omega_{t-1}) = lpha
\]


## 1. Monter Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Installer les dépendances

`arch` est nécessaire pour les modèles GARCH. Les modèles MSM/copules utilisent surtout CPU, donc le GPU Colab n'apporte pas forcément beaucoup de gain. L'intérêt principal ici est de paralléliser avec VSCode/local.

In [ ]:
!pip install arch scipy statsmodels pandas numpy -q

## 3. Créer les modules du projet dans Colab

Cette cellule écrit les fonctions du projet dans `/content/var_project/`. Tu n'as donc pas besoin d'uploader tout le dossier `src/`.

In [ ]:

from pathlib import Path
import sys

PROJECT_DIR = Path('/content/var_project')
SRC_DIR = PROJECT_DIR / 'src'
SRC_DIR.mkdir(parents=True, exist_ok=True)
(SRC_DIR / '__init__.py').write_text('', encoding='utf-8')

MSM_CODE = '"""Markov Switching Multifractal volatility model."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom itertools import product\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\nfrom scipy.stats import norm\n\n\n@dataclass(frozen=True)\nclass MSMParams:\n    """Estimated MSM parameters."""\n\n    m0: float\n    sigma: float\n    b: float\n    gamma_k: float\n    gamma_1: float\n    k: int\n\n\n@dataclass(frozen=True)\nclass MSMFitResult:\n    """Container for MSM estimation results."""\n\n    asset: str\n    k: int\n    params: MSMParams\n    log_likelihood: float\n    success: bool\n    message: str\n    nobs: int\n    mean_return: float\n\n\ndef make_msm_states(k: int, m0: float) -> np.ndarray:\n    """Return all 2^k MSM multiplier states.\n\n    Each component takes value m0 or 2 - m0.\n    """\n    if k < 1:\n        raise ValueError("k must be >= 1.")\n    if not 0.0 < m0 < 2.0:\n        raise ValueError("m0 must lie in (0, 2).")\n\n    values = [m0, 2.0 - m0]\n    states = np.array(list(product(values, repeat=k)), dtype=float)\n    return states\n\n\ndef renewal_probabilities_from_gamma_k(\n    k: int,\n    b: float,\n    gamma_k: float,\n) -> np.ndarray:\n    """Compute gamma_1,...,gamma_k from b and gamma_k.\n\n    The paper reports gamma_k. We invert the usual MSM relation:\n\n        gamma_i = 1 - (1 - gamma_1) ** (b ** (i - 1))\n\n    using gamma_k.\n    """\n    if k < 1:\n        raise ValueError("k must be >= 1.")\n    if b <= 1.0:\n        raise ValueError("b must be > 1.")\n    if not 0.0 < gamma_k < 1.0:\n        raise ValueError("gamma_k must lie in (0, 1).")\n\n    if k == 1:\n        gamma_1 = gamma_k\n    else:\n        gamma_1 = 1.0 - (1.0 - gamma_k) ** (1.0 / (b ** (k - 1)))\n\n    gammas = np.array(\n        [\n            1.0 - (1.0 - gamma_1) ** (b**i)\n            for i in range(k)\n        ],\n        dtype=float,\n    )\n\n    return np.clip(gammas, 1e-10, 1.0 - 1e-10)\n\n\ndef transition_matrix_from_gammas(gammas: np.ndarray) -> np.ndarray:\n    """Build the 2^k x 2^k MSM transition matrix.\n\n    If a component is not renewed, it keeps its current value.\n    If it is renewed, it draws either low or high with probability 1/2.\n    Thus:\n        P(next bit = old bit) = 1 - gamma_i / 2\n        P(next bit = other bit) = gamma_i / 2\n    """\n    gammas = np.asarray(gammas, dtype=float)\n    k = len(gammas)\n\n    bit_states = np.array(list(product([0, 1], repeat=k)), dtype=int)\n    n_states = bit_states.shape[0]\n\n    transition = np.ones((n_states, n_states), dtype=float)\n\n    for i, gamma_i in enumerate(gammas):\n        same = bit_states[:, [i]] == bit_states[:, i][None, :]\n        component_transition = np.where(\n            same,\n            1.0 - gamma_i / 2.0,\n            gamma_i / 2.0,\n        )\n        transition *= component_transition\n\n    # Numerical safety\n    transition /= transition.sum(axis=1, keepdims=True)\n\n    return transition\n\n\ndef msm_loglikelihood(\n    y: pd.Series | np.ndarray,\n    k: int,\n    m0: float,\n    sigma: float,\n    b: float,\n    gamma_k: float,\n) -> float:\n    """Compute MSM log-likelihood by Hamilton filtering."""\n    values = _clean_centered_returns(y)\n\n    if sigma <= 0.0:\n        return -np.inf\n\n    states = make_msm_states(k=k, m0=m0)\n    gammas = renewal_probabilities_from_gamma_k(k=k, b=b, gamma_k=gamma_k)\n    transition = transition_matrix_from_gammas(gammas)\n\n    h = np.sqrt(np.prod(states, axis=1))\n    state_sigmas = sigma * h\n\n    if np.any(state_sigmas <= 0) or np.any(~np.isfinite(state_sigmas)):\n        return -np.inf\n\n    n_states = states.shape[0]\n    # Initial distribution: equal probability for all states.\n    predicted_probs = np.full(n_states, 1.0 / n_states)\n\n    loglik = 0.0\n    log_state_sigmas = np.log(state_sigmas)\n    log_sqrt_2pi = 0.5 * np.log(2.0 * np.pi)\n\n    for obs in values:\n        z = obs / state_sigmas\n        log_state_densities = (\n            -log_sqrt_2pi\n            - log_state_sigmas\n            - 0.5 * z**2\n        )\n\n        log_joint = np.log(predicted_probs + 1e-300) + log_state_densities\n        max_log_joint = np.max(log_joint)\n        log_density = max_log_joint + np.log(np.exp(log_joint - max_log_joint).sum())\n\n        if not np.isfinite(log_density):\n            return -np.inf\n\n        loglik += log_density\n        # Update p_{t|t}\n        filtered_probs = np.exp(log_joint - log_density)\n        # Predict p_{t+1|t}\n        predicted_probs = filtered_probs @ transition\n        predicted_probs = np.clip(predicted_probs, 1e-300, 1.0)\n        predicted_probs /= predicted_probs.sum()\n\n    return float(loglik)\n\n\ndef msm_filter(\n    returns: pd.Series | np.ndarray,\n    k: int,\n    m0: float,\n    sigma: float,\n    b: float,\n    gamma_k: float,\n    mean: float = 0.0,\n    clip_cdf: float = 1e-10,\n) -> dict[str, pd.Series | pd.DataFrame | float]:\n    """Run the MSM Hamilton filter and return conditional objects.\n\n    Parameters\n    ----------\n    returns:\n        Uncentered return series. If you already have centered returns, pass\n        mean=0.0.\n    k:\n        Number of MSM volatility components.\n    m0, sigma, b, gamma_k:\n        MSM parameters.\n    mean:\n        Constant conditional mean used to center returns.\n    clip_cdf:\n        Numerical clipping level for PIT values.\n\n    Returns\n    -------\n    dict\n        Dictionary containing predicted probabilities, filtered probabilities,\n        conditional densities, conditional CDFs, PIT values, conditional\n        volatility and log-likelihood.\n    """\n    series = _as_series_with_index(returns)\n    centered = series - mean\n    values = centered.to_numpy(dtype=float)\n\n    states = make_msm_states(k=k, m0=m0)\n    gammas = renewal_probabilities_from_gamma_k(k=k, b=b, gamma_k=gamma_k)\n    transition = transition_matrix_from_gammas(gammas)\n\n    h = np.sqrt(np.prod(states, axis=1))\n    state_sigmas = sigma * h\n\n    if np.any(state_sigmas <= 0) or np.any(~np.isfinite(state_sigmas)):\n        raise ValueError("Invalid MSM state standard deviations.")\n\n    n_obs = values.shape[0]\n    n_states = states.shape[0]\n\n    predicted_probs = np.full(n_states, 1.0 / n_states)\n\n    predicted_probs_store = np.empty((n_obs, n_states), dtype=float)\n    filtered_probs_store = np.empty((n_obs, n_states), dtype=float)\n    densities = np.empty(n_obs, dtype=float)\n    cdfs = np.empty(n_obs, dtype=float)\n    conditional_volatility = np.empty(n_obs, dtype=float)\n\n    loglik = 0.0\n\n    log_state_sigmas = np.log(state_sigmas)\n    log_sqrt_2pi = 0.5 * np.log(2.0 * np.pi)\n\n    for t, obs in enumerate(values):\n        predicted_probs_store[t, :] = predicted_probs\n        z = obs / state_sigmas\n        log_state_densities = (\n            -log_sqrt_2pi\n            - log_state_sigmas\n            - 0.5 * z**2\n        )\n        log_joint = np.log(predicted_probs + 1e-300) + log_state_densities\n        max_log_joint = np.max(log_joint)\n        log_density = max_log_joint + np.log(\n            np.exp(log_joint - max_log_joint).sum()\n        )\n        density_t = float(np.exp(log_density))\n        densities[t] = density_t\n        loglik += float(log_density)\n        # Conditional CDF:\n        # F(y_t | Omega_{t-1}) = sum_j p_{t|t-1}(j) Phi(y_t / sigma_j)\n        state_cdfs = norm.cdf(z)\n        cdf_t = float(np.sum(predicted_probs * state_cdfs))\n        cdfs[t] = np.clip(cdf_t, clip_cdf, 1.0 - clip_cdf)\n        # Conditional volatility:\n        # sqrt(E[sigma_t^2 | Omega_{t-1}])\n        conditional_variance_t = float(\n            np.sum(predicted_probs * state_sigmas**2)\n        )\n        conditional_volatility[t] = np.sqrt(conditional_variance_t)\n        # Bayesian update:\n        # p_{t|t}(j) = p_{t|t-1}(j) f_j(y_t) / f(y_t | Omega_{t-1})\n        filtered_probs = np.exp(log_joint - log_density)\n        filtered_probs_store[t, :] = filtered_probs\n        # Prediction:\n        # p_{t+1|t} = p_{t|t} P\n        predicted_probs = filtered_probs @ transition\n        predicted_probs = np.clip(predicted_probs, 1e-300, 1.0)\n        predicted_probs /= predicted_probs.sum()\n\n    state_columns = [f"state_{i}" for i in range(n_states)]\n\n    return {\n        "centered_returns": pd.Series(\n            centered.to_numpy(),\n            index=series.index,\n            name=series.name,\n        ),\n        "predicted_probs": pd.DataFrame(\n            predicted_probs_store,\n            index=series.index,\n            columns=state_columns,\n        ),\n        "filtered_probs": pd.DataFrame(\n            filtered_probs_store,\n            index=series.index,\n            columns=state_columns,\n        ),\n        "densities": pd.Series(\n            densities,\n            index=series.index,\n            name=series.name,\n        ),\n        "cdfs": pd.Series(\n            cdfs,\n            index=series.index,\n            name=series.name,\n        ),\n        "pit": pd.Series(\n            cdfs,\n            index=series.index,\n            name=series.name,\n        ),\n        "conditional_volatility": pd.Series(\n            conditional_volatility,\n            index=series.index,\n            name=series.name,\n        ),\n        "log_likelihood": float(loglik),\n        "states": pd.DataFrame(states, columns=[f"M_{i+1}" for i in range(k)]),\n        "gammas": pd.Series(gammas, index=[f"gamma_{i+1}" for i in range(k)]),\n        "state_sigmas": pd.Series(state_sigmas, index=state_columns),\n    }\n\n\ndef msm_conditional_cdf(\n    returns: pd.Series | np.ndarray,\n    k: int,\n    m0: float,\n    sigma: float,\n    b: float,\n    gamma_k: float,\n    mean: float = 0.0,\n    clip_cdf: float = 1e-10,\n) -> pd.Series:\n    """Return the MSM conditional CDF values F(y_t | Omega_{t-1})."""\n    filtered = msm_filter(\n        returns=returns,\n        k=k,\n        m0=m0,\n        sigma=sigma,\n        b=b,\n        gamma_k=gamma_k,\n        mean=mean,\n        clip_cdf=clip_cdf,\n    )\n    return filtered["cdfs"]\n\n\ndef msm_probability_integral_transform(\n    returns: pd.Series,\n    fit_result: MSMFitResult,\n    clip_cdf: float = 1e-10,\n) -> pd.Series:\n    """Compute MSM PIT values from an estimated MSMFitResult.\n\n    The output is:\n        u_t = F_MSM(r_t - mu_hat | Omega_{t-1})\n    """\n    params = fit_result.params\n\n    pit = msm_conditional_cdf(\n        returns=returns,\n        k=fit_result.k,\n        m0=params.m0,\n        sigma=params.sigma,\n        b=params.b,\n        gamma_k=params.gamma_k,\n        mean=fit_result.mean_return,\n        clip_cdf=clip_cdf,\n    )\n\n    pit.name = returns.name\n    return pit\n\n\ndef msm_filter_from_result(\n    returns: pd.Series,\n    fit_result: MSMFitResult,\n    clip_cdf: float = 1e-10,\n) -> dict[str, pd.Series | pd.DataFrame | float]:\n    """Run the MSM filter using a stored MSMFitResult."""\n    params = fit_result.params\n\n    return msm_filter(\n        returns=returns,\n        k=fit_result.k,\n        m0=params.m0,\n        sigma=params.sigma,\n        b=params.b,\n        gamma_k=params.gamma_k,\n        mean=fit_result.mean_return,\n        clip_cdf=clip_cdf,\n    )\n\n\ndef build_msm_pit_frame(\n    returns: pd.DataFrame,\n    fit_results: dict[str, MSMFitResult],\n    clip_cdf: float = 1e-10,\n) -> pd.DataFrame:\n    """Build a DataFrame of MSM PIT values for several assets."""\n    pits = {}\n\n    for asset in returns.columns:\n        if asset not in fit_results:\n            raise KeyError(f"Missing MSM fit result for asset: {asset}")\n\n        pits[asset] = msm_probability_integral_transform(\n            returns=returns[asset],\n            fit_result=fit_results[asset],\n            clip_cdf=clip_cdf,\n        )\n\n    pit_frame = pd.concat(pits, axis=1).dropna(how="any")\n    pit_frame.index.name = returns.index.name or "date"\n\n    return pit_frame\n\n\ndef _as_series_with_index(values: pd.Series | np.ndarray) -> pd.Series:\n    """Convert array-like values to a clean Series while preserving index if possible."""\n    if isinstance(values, pd.Series):\n        series = pd.to_numeric(values, errors="coerce").dropna()\n        return series.astype(float)\n\n    array = np.asarray(values, dtype=float)\n    array = array[np.isfinite(array)]\n\n    if array.size == 0:\n        raise ValueError("At least one observation is required.")\n\n    return pd.Series(array, dtype=float)\n\n\ndef fit_msm(\n    returns: pd.Series,\n    k: int,\n    n_starts: int = 20,\n    seed: int = 123,\n    verbose: bool = True,\n) -> MSMFitResult:\n    """Estimate MSM parameters for one return series and one k."""\n    series = pd.to_numeric(returns, errors="coerce").dropna()\n    asset = str(series.name or "asset")\n\n    # The MSM is estimated on centered percentage returns.\n    mean_return = float(series.mean())\n    y = series - mean_return\n    rng = np.random.default_rng(seed)\n    best_result = None\n    best_loglik = -np.inf\n    bounds = [\n        (0.05, 1.95),   # m0\n        (1e-4, 10.0),   # sigma\n        (1.0001, 50.0), # b\n        (1e-5, 0.999),  # gamma_k\n    ]\n    initial_points = _initial_points_for_msm(\n        y=y,\n        k=k,\n        n_starts=n_starts,\n        rng=rng,\n    )\n\n    y_array = y.to_numpy(dtype=float)\n    for start_id, x0 in enumerate(initial_points, start=1):\n        if verbose:\n            print(f"  start {start_id}/{len(initial_points)}: x0={x0}")\n        opt = minimize(\n            _negative_loglikelihood,\n            x0=x0,\n            args=(y_array, k),\n            method="L-BFGS-B",\n            bounds=bounds,\n            options={\n                "maxiter": 300,\n                "maxls": 20,\n                "ftol": 1e-6,\n                "gtol": 1e-5,\n            },\n        )\n        loglik = -float(opt.fun)\n        if verbose:\n            print(\n                f"    success={opt.success}, "\n                f"loglik={loglik:.3f}, "\n                f"nit={getattr(opt, \'nit\', None)}, "\n                f"nfev={getattr(opt, \'nfev\', None)}"\n            )\n        if np.isfinite(loglik) and loglik > best_loglik:\n            best_loglik = loglik\n            best_result = opt\n\n    if best_result is None:\n        raise RuntimeError(f"MSM estimation failed for {asset}, k={k}.")\n\n    m0, sigma, b, gamma_k = best_result.x\n    gammas = renewal_probabilities_from_gamma_k(k=k, b=b, gamma_k=gamma_k)\n\n    params = MSMParams(\n        m0=float(m0),\n        sigma=float(sigma),\n        b=float(b),\n        gamma_k=float(gamma_k),\n        gamma_1=float(gammas[0]),\n        k=k,\n    )\n\n    return MSMFitResult(\n        asset=asset,\n        k=k,\n        params=params,\n        log_likelihood=float(best_loglik),\n        success=bool(best_result.success),\n        message=str(best_result.message),\n        nobs=int(series.shape[0]),\n        mean_return=mean_return,\n    )\n\n\ndef fit_msm_grid(\n    returns: pd.DataFrame,\n    k_values: range | list[int] = range(1, 8),\n    n_starts: int = 20,\n    seed: int = 123,\n    verbose: bool = True,\n) -> pd.DataFrame:\n    """Estimate MSM for each asset and each k, then return a comparison table."""\n    rows = []\n\n    for asset in returns.columns:\n        for k in k_values:\n            if verbose:\n                print(f"Estimating MSM: asset={asset}, k={k}, n_starts={n_starts}")\n            result = fit_msm(\n                returns=returns[asset],\n                k=int(k),\n                n_starts=n_starts,\n                seed=seed + 1000 * int(k),\n                verbose=verbose,\n            )\n\n            rows.append(msm_fit_result_to_dict(result))\n\n    return pd.DataFrame(rows)\n\n\ndef msm_fit_result_to_dict(\n    result: MSMFitResult,\n) -> dict[str, float | int | str | bool]:\n    """Convert a MSMFitResult to a flat dictionary for tables."""\n    return {\n        "asset": result.asset,\n        "k": result.k,\n        "mean_return": result.mean_return,\n        "m0": result.params.m0,\n        "sigma": result.params.sigma,\n        "b": result.params.b,\n        "gamma_1": result.params.gamma_1,\n        "gamma_k": result.params.gamma_k,\n        "log_likelihood": result.log_likelihood,\n        "success": result.success,\n        "message": result.message,\n        "nobs": result.nobs,\n    }\n\n\ndef _negative_loglikelihood(\n    params: np.ndarray,\n    y: np.ndarray,\n    k: int,\n) -> float:\n    m0, sigma, b, gamma_k = params\n\n    if not (\n        0.0 < m0 < 2.0\n        and sigma > 0.0\n        and b > 1.0\n        and 0.0 < gamma_k < 1.0\n    ):\n        return 1e50\n\n    loglik = msm_loglikelihood(\n        y=y,\n        k=k,\n        m0=float(m0),\n        sigma=float(sigma),\n        b=float(b),\n        gamma_k=float(gamma_k),\n    )\n\n    if not np.isfinite(loglik):\n        return 1e50\n\n    return -float(loglik)\n\n\ndef _initial_points_for_msm(\n    y: pd.Series,\n    k: int,\n    n_starts: int,\n    rng: np.random.Generator,\n) -> list[np.ndarray]:\n    """Generate exactly n_starts starting values for numerical optimization."""\n    sample_sigma = float(y.std(ddof=1))\n    sample_sigma = max(sample_sigma, 1e-2)\n\n    deterministic = [\n        np.array([1.50, sample_sigma, 2.0, 0.10]),\n        np.array([1.50, sample_sigma, 5.0, 0.20]),\n        np.array([1.40, sample_sigma, 10.0, 0.10]),\n        np.array([1.60, sample_sigma, 10.0, 0.20]),\n        np.array([1.30, sample_sigma, 20.0, 0.10]),\n    ]\n\n    points = deterministic[: min(n_starts, len(deterministic))]\n\n    while len(points) < n_starts:\n        points.append(\n            np.array(\n                [\n                    rng.uniform(1.1, 1.8),\n                    sample_sigma * rng.uniform(0.6, 1.6),\n                    rng.uniform(1.2, 30.0),\n                    rng.uniform(0.02, 0.95),\n                ],\n                dtype=float,\n            )\n        )\n\n    return points\n\n\ndef _clean_centered_returns(y: pd.Series | np.ndarray) -> np.ndarray:\n    values = np.asarray(y, dtype=float)\n    values = values[np.isfinite(values)]\n\n    if values.size == 0:\n        raise ValueError("MSM estimation requires at least one observation.")\n\n    return values'
COPULAS_CODE = '"""Bivariate copula estimation for copula-MSM and copula-GARCH models."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import minimize\nfrom scipy.stats import multivariate_normal, multivariate_t, norm, t\n\n\nEPS = 1e-10\n\n\n@dataclass(frozen=True)\nclass CopulaFitResult:\n    """Container for an estimated bivariate copula."""\n\n    copula: str\n    margin_model: str\n    params: dict[str, float]\n    log_likelihood: float\n    aic: float\n    bic: float\n    nobs: int\n    success: bool\n    message: str\n\n\ndef _validate_uniforms(uniforms: pd.DataFrame) -> pd.DataFrame:\n    """Validate and clip a two-column PIT DataFrame."""\n    if uniforms.shape[1] != 2:\n        raise ValueError("Copula estimation expects exactly two uniform series.")\n\n    frame = uniforms.copy()\n    frame = frame.apply(pd.to_numeric, errors="coerce").dropna(how="any")\n    frame = frame.clip(lower=EPS, upper=1.0 - EPS)\n\n    if frame.empty:\n        raise ValueError("At least one observation is required.")\n\n    return frame\n\n\ndef gaussian_copula_logpdf(uniforms: pd.DataFrame, rho: float) -> np.ndarray:\n    """Gaussian copula log-density."""\n    frame = _validate_uniforms(uniforms)\n\n    if not -0.999 < rho < 0.999:\n        return np.full(len(frame), -np.inf)\n\n    z = norm.ppf(frame.to_numpy(dtype=float))\n    corr = np.array([[1.0, rho], [rho, 1.0]])\n\n    joint = multivariate_normal.logpdf(\n        z,\n        mean=np.zeros(2),\n        cov=corr,\n    )\n\n    marginals = norm.logpdf(z[:, 0]) + norm.logpdf(z[:, 1])\n    return joint - marginals\n\n\ndef student_copula_logpdf(\n    uniforms: pd.DataFrame,\n    rho: float,\n    nu: float,\n) -> np.ndarray:\n    """Student-t copula log-density."""\n    frame = _validate_uniforms(uniforms)\n\n    if not -0.999 < rho < 0.999 or nu <= 2.01:\n        return np.full(len(frame), -np.inf)\n\n    x = t.ppf(frame.to_numpy(dtype=float), df=nu)\n    shape = np.array([[1.0, rho], [rho, 1.0]])\n\n    joint = multivariate_t.logpdf(\n        x,\n        loc=np.zeros(2),\n        shape=shape,\n        df=nu,\n    )\n\n    marginals = t.logpdf(x[:, 0], df=nu) + t.logpdf(x[:, 1], df=nu)\n    return joint - marginals\n\n\ndef clayton_copula_logpdf(uniforms: pd.DataFrame, theta: float) -> np.ndarray:\n    """Clayton copula log-density."""\n    frame = _validate_uniforms(uniforms)\n\n    if theta <= 0:\n        return np.full(len(frame), -np.inf)\n\n    u = frame.iloc[:, 0].to_numpy(dtype=float)\n    v = frame.iloc[:, 1].to_numpy(dtype=float)\n\n    s = u ** (-theta) + v ** (-theta) - 1.0\n\n    log_density = (\n        np.log(theta + 1.0)\n        + (-theta - 1.0) * (np.log(u) + np.log(v))\n        + (-2.0 - 1.0 / theta) * np.log(s)\n    )\n\n    return log_density\n\n\ndef rotated_clayton_copula_logpdf(\n    uniforms: pd.DataFrame,\n    theta: float,\n) -> np.ndarray:\n    """Survival, or 180-degree rotated, Clayton copula log-density."""\n    frame = _validate_uniforms(uniforms)\n    rotated = 1.0 - frame\n    return clayton_copula_logpdf(rotated, theta=theta)\n\n\ndef gumbel_copula_logpdf(uniforms: pd.DataFrame, theta: float) -> np.ndarray:\n    """Gumbel copula log-density."""\n    frame = _validate_uniforms(uniforms)\n\n    if theta < 1.0:\n        return np.full(len(frame), -np.inf)\n\n    u = frame.iloc[:, 0].to_numpy(dtype=float)\n    v = frame.iloc[:, 1].to_numpy(dtype=float)\n\n    x = -np.log(u)\n    y = -np.log(v)\n\n    a = x**theta + y**theta\n    s = a ** (1.0 / theta)\n\n    log_density = (\n        -s\n        - np.log(u)\n        - np.log(v)\n        + (theta - 1.0) * (np.log(x) + np.log(y))\n        + (1.0 / theta - 2.0) * np.log(a)\n        + np.log(s + theta - 1.0)\n    )\n\n    return log_density\n\n\ndef rotated_gumbel_copula_logpdf(\n    uniforms: pd.DataFrame,\n    theta: float,\n) -> np.ndarray:\n    """Survival, or 180-degree rotated, Gumbel copula log-density."""\n    frame = _validate_uniforms(uniforms)\n    rotated = 1.0 - frame\n    return gumbel_copula_logpdf(rotated, theta=theta)\n\n\ndef frank_copula_logpdf(uniforms: pd.DataFrame, theta: float) -> np.ndarray:\n    """Frank copula log-density with numerical safeguards."""\n    frame = _validate_uniforms(uniforms)\n\n    if abs(theta) < 1e-6:\n        return np.zeros(len(frame))\n\n    u = frame.iloc[:, 0].to_numpy(dtype=float)\n    v = frame.iloc[:, 1].to_numpy(dtype=float)\n\n    with np.errstate(over="ignore", under="ignore", divide="ignore", invalid="ignore"):\n        numerator = (\n            theta\n            * (1.0 - np.exp(-theta))\n            * np.exp(-theta * (u + v))\n        )\n\n        denominator = (\n            (1.0 - np.exp(-theta))\n            - (1.0 - np.exp(-theta * u)) * (1.0 - np.exp(-theta * v))\n        ) ** 2\n\n        density = numerator / denominator\n\n    if np.any(~np.isfinite(density)) or np.any(density <= 0):\n        return np.full(len(frame), -np.inf)\n\n    return np.log(density)\n\n\ndef plackett_copula_logpdf(uniforms: pd.DataFrame, theta: float) -> np.ndarray:\n    """Plackett copula log-density.\n\n    theta = 1 corresponds to independence.\n    """\n    frame = _validate_uniforms(uniforms)\n\n    if theta <= 0:\n        return np.full(len(frame), -np.inf)\n\n    if abs(theta - 1.0) < 1e-6:\n        return np.zeros(len(frame))\n\n    u = frame.iloc[:, 0].to_numpy(dtype=float)\n    v = frame.iloc[:, 1].to_numpy(dtype=float)\n\n    a = theta - 1.0\n    denominator_base = (\n        (1.0 + a * (u + v)) ** 2\n        - 4.0 * theta * a * u * v\n    )\n\n    numerator = theta * (1.0 + a * (u + v - 2.0 * u * v))\n    if np.any(denominator_base <= 0) or np.any(~np.isfinite(denominator_base)):\n        return np.full(len(frame), -np.inf)\n\n    density = numerator / (denominator_base ** 1.5)\n\n    if np.any(~np.isfinite(density)) or np.any(density <= 0):\n        return np.full(len(frame), -np.inf)\n\n    return np.log(density)\n\n\ndef _jc_logpdf(\n    u: np.ndarray,\n    v: np.ndarray,\n    tau_u: float,\n    tau_l: float,\n) -> np.ndarray:\n    """Log-density of the Joe-Clayton (BB7) copula.\n\n    Patton (2006), Appendix A.\n    kappa = 1 / log2(2 - tau_u), gamma = -1 / log2(tau_l).\n    """\n    kappa = 1.0 / np.log2(2.0 - tau_u)\n    gamma = -1.0 / np.log2(tau_l)\n\n    a = np.clip(1.0 - (1.0 - u) ** kappa, EPS, 1.0 - EPS)\n    b = np.clip(1.0 - (1.0 - v) ** kappa, EPS, 1.0 - EPS)\n\n    s = a ** (-gamma) + b ** (-gamma) - 1.0\n    s = np.maximum(s, 1.0 + EPS)\n\n    s_neg_q = s ** (-1.0 / gamma)\n    one_minus = np.clip(1.0 - s_neg_q, EPS, 1.0)\n\n    bracket = (gamma + 1.0) / gamma - (kappa * gamma + 1.0) / (kappa * gamma) * s_neg_q\n\n    log_density = (\n        np.log(kappa)\n        + np.log(gamma)\n        + (1.0 / kappa - 2.0) * np.log(one_minus)\n        + (-1.0 / gamma - 2.0) * np.log(s)\n        + np.log(np.maximum(bracket, EPS))\n        + (-gamma - 1.0) * (np.log(a) + np.log(b))\n        + (kappa - 1.0) * (np.log1p(-(u)) + np.log1p(-(v)))\n    )\n\n    return log_density\n\n\ndef sjc_copula_logpdf(\n    uniforms: pd.DataFrame,\n    tau_u: float,\n    tau_l: float,\n) -> np.ndarray:\n    """Symmetrized Joe-Clayton copula log-density (Patton 2006).\n\n    tau_u: upper tail dependence in (0, 1).\n    tau_l: lower tail dependence in (0, 1).\n    c_SJC = 0.5 * (c_JC(u,v; tau_u, tau_l) + c_JC(1-u, 1-v; tau_l, tau_u))\n    """\n    frame = _validate_uniforms(uniforms)\n\n    if not (0.0 < tau_u < 1.0) or not (0.0 < tau_l < 1.0):\n        return np.full(len(frame), -np.inf)\n\n    u = frame.iloc[:, 0].to_numpy(dtype=float)\n    v = frame.iloc[:, 1].to_numpy(dtype=float)\n\n    log_c1 = _jc_logpdf(u, v, tau_u, tau_l)\n    log_c2 = _jc_logpdf(1.0 - u, 1.0 - v, tau_l, tau_u)\n\n    log_max = np.maximum(log_c1, log_c2)\n    log_sum = log_max + np.log(np.exp(log_c1 - log_max) + np.exp(log_c2 - log_max))\n\n    return np.log(0.5) + log_sum\n\n\ndef _copula_logpdf_from_vector(\n    copula: str,\n    uniforms: pd.DataFrame,\n    x: np.ndarray,\n) -> np.ndarray:\n    """Dispatch log-density evaluation from an optimizer parameter vector."""\n    copula = copula.lower()\n\n    if copula == "gaussian":\n        return gaussian_copula_logpdf(uniforms, rho=float(x[0]))\n\n    if copula == "student":\n        return student_copula_logpdf(\n            uniforms,\n            rho=float(x[0]),\n            nu=float(x[1]),\n        )\n\n    if copula == "plackett":\n        return plackett_copula_logpdf(uniforms, theta=float(x[0]))\n\n    if copula == "clayton":\n        return clayton_copula_logpdf(uniforms, theta=float(x[0]))\n\n    if copula == "rotated_clayton":\n        return rotated_clayton_copula_logpdf(uniforms, theta=float(x[0]))\n    \n    if copula == "sjc":\n        return sjc_copula_logpdf(uniforms, tau_u=float(x[0]), tau_l=float(x[1]))\n\n    if copula == "frank":\n        return frank_copula_logpdf(uniforms, theta=float(x[0]))\n\n    if copula == "gumbel":\n        return gumbel_copula_logpdf(uniforms, theta=float(x[0]))\n\n    if copula == "rotated_gumbel":\n        return rotated_gumbel_copula_logpdf(uniforms, theta=float(x[0]))\n\n    raise ValueError(f"Unknown copula: {copula}")\n\n\ndef _negative_loglikelihood(\n    x: np.ndarray,\n    copula: str,\n    uniforms: pd.DataFrame,\n) -> float:\n    logpdf = _copula_logpdf_from_vector(copula, uniforms, x)\n\n    if np.any(~np.isfinite(logpdf)):\n        return 1e50\n\n    return -float(np.sum(logpdf))\n\n\ndef _initial_points_and_bounds(\n    copula: str,\n    uniforms: pd.DataFrame,\n) -> tuple[list[np.ndarray], list[tuple[float, float]]]:\n    """Return starting values and bounds for a copula."""\n    frame = _validate_uniforms(uniforms)\n    z = norm.ppf(frame.to_numpy(dtype=float))\n    rho0 = float(np.corrcoef(z[:, 0], z[:, 1])[0, 1])\n    rho0 = float(np.clip(rho0, -0.95, 0.95))\n\n    copula = copula.lower()\n\n    if copula == "gaussian":\n        return [np.array([rho0])], [(-0.999, 0.999)]\n\n    if copula == "student":\n        return [\n            np.array([rho0, 5.0]),\n            np.array([rho0, 10.0]),\n            np.array([rho0, 20.0]),\n        ], [(-0.999, 0.999), (2.01, 100.0)]\n    \n    if copula == "plackett":\n        return [\n            np.array([20.0]),\n            np.array([50.0]),\n            np.array([80.0]),\n            np.array([150.0]),\n        ], [(1e-4, 500.0)]\n\n    if copula in {"clayton", "rotated_clayton"}:\n        return [\n            np.array([1.0]),\n            np.array([2.0]),\n            np.array([4.0]),\n            np.array([8.0]),\n        ], [(1e-4, 50.0)]\n\n    if copula == "frank":\n        return [\n            np.array([5.0]),\n            np.array([10.0]),\n            np.array([20.0]),\n            np.array([30.0]),\n            np.array([50.0]),\n        ], [(-100.0, 100.0)]\n\n    if copula in {"gumbel", "rotated_gumbel"}:\n        return [\n            np.array([1.2]),\n            np.array([2.0]),\n            np.array([4.0]),\n            np.array([6.0]),\n            np.array([10.0]),\n        ], [(1.0001, 50.0)]\n\n    if copula == "sjc":\n        bounds = [(1e-4, 0.9999), (1e-4, 0.9999)]\n        return [\n            np.array([0.10, 0.10]),\n            np.array([0.20, 0.20]),\n            np.array([0.30, 0.10]),\n            np.array([0.10, 0.30]),\n            np.array([0.30, 0.30]),\n        ], bounds\n\n    raise ValueError(f"Unknown copula: {copula}")\n\n\ndef _params_dict(copula: str, x: np.ndarray) -> dict[str, float]:\n    copula = copula.lower()\n\n    if copula == "gaussian":\n        return {"rho": float(x[0])}\n\n    if copula == "student":\n        return {"rho": float(x[0]), "nu": float(x[1])}\n\n    if copula == "plackett":\n        return {"theta": float(x[0])}\n\n    if copula in {"clayton", "rotated_clayton"}:\n        return {"theta": float(x[0])}\n    \n    if copula == "sjc":\n        return {"tau_u": float(x[0]), "tau_l": float(x[1])}\n\n    if copula == "frank":\n        return {"theta": float(x[0])}\n\n    if copula in {"gumbel", "rotated_gumbel"}:\n        return {"theta": float(x[0])}\n\n    raise ValueError(f"Unknown copula: {copula}")\n\n\ndef fit_copula(\n    uniforms: pd.DataFrame,\n    copula: str,\n    margin_model: str,\n) -> CopulaFitResult:\n    """Estimate one bivariate copula by maximum likelihood."""\n    frame = _validate_uniforms(uniforms)\n\n    initial_points, bounds = _initial_points_and_bounds(copula, frame)\n\n    best_opt = None\n    best_fun = np.inf\n    fallback_opt = None\n    fallback_fun = np.inf\n\n    for x0 in initial_points:\n        opt = minimize(\n            _negative_loglikelihood,\n            x0=x0,\n            args=(copula, frame),\n            method="L-BFGS-B",\n            bounds=bounds,\n            options={"maxiter": 3000, "ftol": 1e-10},\n        )\n        if np.isfinite(opt.fun) and opt.fun < fallback_fun:\n            fallback_fun = float(opt.fun)\n            fallback_opt = opt\n        if opt.success and np.isfinite(opt.fun) and opt.fun < best_fun:\n            best_fun = float(opt.fun)\n            best_opt = opt\n\n    if best_opt is None:\n        best_opt = fallback_opt\n    if best_opt is None:\n        raise RuntimeError(f"Copula estimation failed for {copula}.")\n\n    log_likelihood = -float(best_opt.fun)\n    n_params = len(best_opt.x)\n    nobs = int(frame.shape[0])\n\n    return CopulaFitResult(\n        copula=copula,\n        margin_model=margin_model,\n        params=_params_dict(copula, best_opt.x),\n        log_likelihood=log_likelihood,\n        aic=-2.0 * log_likelihood + 2.0 * n_params,\n        bic=-2.0 * log_likelihood + np.log(nobs) * n_params,\n        nobs=nobs,\n        success=bool(best_opt.success),\n        message=str(best_opt.message),\n    )\n\n\ndef fit_all_copulas(\n    uniforms: pd.DataFrame,\n    margin_model: str,\n    copulas: tuple[str, ...] = (\n        "gaussian",\n        "student",\n        "plackett",\n        "clayton",\n        "rotated_clayton",\n        "frank",\n        "gumbel",\n        "rotated_gumbel",\n        "sjc",\n    ),\n) -> list[CopulaFitResult]:\n    """Estimate several copulas on the same PIT data."""\n    results = []\n\n    for copula in copulas:\n        result = fit_copula(\n            uniforms=uniforms,\n            copula=copula,\n            margin_model=margin_model,\n        )\n        results.append(result)\n\n    return results\n\n\ndef fit_copula_grid(\n    pit_by_model: dict[str, pd.DataFrame],\n    copulas: tuple[str, ...] = (\n        "gaussian",\n        "student",\n        "plackett",\n        "clayton",\n        "rotated_clayton",\n        "sjc",\n        "frank",\n        "gumbel",\n        "rotated_gumbel",\n    ),\n) -> list[CopulaFitResult]:\n    """Estimate copulas for several margin models, e.g. MSM and GARCH."""\n    results = []\n\n    for margin_model, uniforms in pit_by_model.items():\n        results.extend(\n            fit_all_copulas(\n                uniforms=uniforms,\n                margin_model=margin_model,\n                copulas=copulas,\n            )\n        )\n\n    return results\n\n\ndef copula_fit_result_to_dict(\n    result: CopulaFitResult,\n) -> dict[str, float | int | str | bool]:\n    """Convert a CopulaFitResult to a flat dictionary."""\n    row: dict[str, float | int | str | bool] = {\n        "margin_model": result.margin_model,\n        "copula": result.copula,\n        "log_likelihood": result.log_likelihood,\n        "aic": result.aic,\n        "bic": result.bic,\n        "nobs": result.nobs,\n        "success": result.success,\n        "message": result.message,\n    }\n\n    row.update(result.params)\n    return row\n\n\ndef copula_results_table(\n    results: list[CopulaFitResult],\n) -> pd.DataFrame:\n    """Return a table of copula estimates."""\n    return pd.DataFrame(\n        [copula_fit_result_to_dict(result) for result in results]\n    )\n\n\ndef format_copula_table_4(table: pd.DataFrame) -> pd.DataFrame:\n    """Format copula estimates close to the paper\'s Table 4."""\n    rows = []\n\n    for _, row in table.iterrows():\n        params = []\n\n        for name in ["rho", "nu", "theta", "tau_u", "tau_l"]:\n            if name in row and pd.notna(row[name]):\n                params.append(f"{name}={row[name]:.3f}")\n\n        rows.append(\n            {\n                "margin_model": row["margin_model"],\n                "copula": row["copula"],\n                "parameters": ", ".join(params),\n                "Log(L)": f"{row[\'log_likelihood\']:.3f}",\n                "AIC": f"{row[\'aic\']:.3f}",\n                "BIC": f"{row[\'bic\']:.3f}",\n            }\n        )\n\n    return pd.DataFrame(rows)\n\n\ndef pseudo_observations(data: pd.DataFrame | pd.Series) -> pd.DataFrame:\n    """Convert observations to empirical CDF ranks in the open unit interval."""\n    frame = data.to_frame() if isinstance(data, pd.Series) else data\n    ranks = frame.rank(method="average", pct=False)\n    return ranks / (len(frame) + 1.0)\n\n\ndef gaussian_copula_correlation(uniforms: pd.DataFrame) -> pd.DataFrame:\n    """Estimate a Gaussian copula correlation matrix from pseudo-observations."""\n    if not ((uniforms > 0) & (uniforms < 1)).all().all():\n        raise ValueError("Gaussian copula inputs must lie strictly between 0 and 1.")\n    normal_scores = pd.DataFrame(\n        norm.ppf(uniforms),\n        index=uniforms.index,\n        columns=uniforms.columns,\n    )\n    return normal_scores.corr()\n\n\ndef simulate_gaussian_copula(\n    correlation: pd.DataFrame | np.ndarray,\n    n_samples: int,\n    seed: int | None = None,\n) -> np.ndarray:\n    """Simulate uniforms from a Gaussian copula."""\n    rng = np.random.default_rng(seed)\n    corr = np.asarray(correlation, dtype=float)\n    draws = rng.multivariate_normal(np.zeros(corr.shape[0]), corr, size=n_samples)\n    return norm.cdf(draws)'
VAR_V2_CODE = '"""Paper-like VaR forecasting helpers for Segnon & Trede copula-MSM replication.\n\nConventions\n-----------\n- Inputs are percentage log returns: 100 * log(P_t/P_{t-1}).\n- VaR is a signed return quantile, usually negative:\n      P(r_p,t <= VaR_t(alpha) | Omega_{t-1}) = alpha\n- Rolling evaluation uses a fixed estimation window, e.g. 1135 observations,\n  and forecasts the next 500 one-day-ahead VaR values.\n\nMain paper-like functions\n-------------------------\n- forecast_historical_var_rolling\n- forecast_variance_covariance_var_rolling\n- forecast_riskmetrics_var_rolling\n- forecast_ccc_garch_var_rolling\n- forecast_garch_copula_var_rolling\n- forecast_msm_copula_var_rolling\n- forecast_all_var_models\n\nNotes\n-----\nThe copula portfolio CDF is evaluated through the identity\n\n    F_p(q) = integral_0^1 P(U1 <= F1((q - (1-pi)F2^{-1}(u2))/pi) | U2=u2) du2.\n\nFor Gaussian and Student copulas, the conditional CDF is analytic. For Plackett,\nClayton, rotated Clayton, Frank, Gumbel, rotated Gumbel, and SJC, it is computed\nas a stable finite-difference derivative of the copula CDF with respect to u2.\nThis is slower but directly matches the paper\'s numerical-integration approach.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Callable, Iterable\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.optimize import brentq\nfrom scipy.stats import multivariate_normal, multivariate_t, norm, t\n\nEPS = 1e-10\n\nSUPPORTED_COPULAS = (\n    "gaussian",\n    "student",\n    "plackett",\n    "clayton",\n    "rotated_clayton",\n    "sjc",\n    "frank",\n    "gumbel",\n    "rotated_gumbel",\n)\n\n\n@dataclass(frozen=True)\nclass RollingSpec:\n    """Rolling-window VaR evaluation specification."""\n\n    n_insample: int = 1135\n    n_oos: int = 500\n    alpha: float = 0.05\n    weights: tuple[float, float] = (0.5, 0.5)\n\n\n# -----------------------------------------------------------------------------\n# Generic validation and rolling-window helpers\n# -----------------------------------------------------------------------------\n\n\ndef prepare_bivariate_returns(\n    returns: pd.DataFrame,\n    n_insample: int = 1135,\n    n_oos: int = 500,\n) -> pd.DataFrame:\n    """Clean and truncate bivariate returns to the paper\'s rolling sample length."""\n    frame = returns.apply(pd.to_numeric, errors="coerce").dropna(how="any")\n    if frame.shape[1] != 2:\n        raise ValueError("The paper replication expects exactly two return columns.")\n\n    required = n_insample + n_oos\n    if len(frame) < required:\n        raise ValueError(f"Need at least {required} observations, got {len(frame)}.")\n\n    # Use the first required observations to reproduce the 1135/500 split exactly.\n    return frame.iloc[:required].copy()\n\n\ndef validate_alpha_weights(alpha: float, weights: Iterable[float]) -> np.ndarray:\n    if not 0.0 < alpha < 1.0:\n        raise ValueError("alpha must be in (0, 1).")\n    w = np.asarray(tuple(weights), dtype=float)\n    if w.shape != (2,):\n        raise ValueError("weights must contain exactly two entries.")\n    if np.any(w < 0.0) or not np.isclose(w.sum(), 1.0):\n        raise ValueError("weights must be non-negative and sum to 1.")\n    return w\n\n\ndef rolling_windows(frame: pd.DataFrame, n_insample: int, n_oos: int):\n    """Yield (i, forecast_date, estimation_window) for a fixed-size rolling scheme."""\n    frame = prepare_bivariate_returns(frame, n_insample=n_insample, n_oos=n_oos)\n    for i in range(n_oos):\n        start = i\n        end = i + n_insample\n        yield i, frame.index[end], frame.iloc[start:end]\n\n\n# -----------------------------------------------------------------------------\n# Benchmark VaR methods from Section 3.3 / Section 4.3\n# -----------------------------------------------------------------------------\n\n\ndef forecast_historical_var_rolling(\n    returns: pd.DataFrame,\n    alpha: float = 0.05,\n    weights: Iterable[float] = (0.5, 0.5),\n    n_insample: int = 1135,\n    n_oos: int = 500,\n) -> pd.Series:\n    """Rolling historical simulation VaR of portfolio returns."""\n    w = validate_alpha_weights(alpha, weights)\n    frame = prepare_bivariate_returns(returns, n_insample, n_oos)\n    portfolio = frame @ w\n    values, dates = [], []\n\n    for i in range(n_oos):\n        end = i + n_insample\n        window = portfolio.iloc[i:end]\n        values.append(float(np.quantile(window, alpha)))\n        dates.append(portfolio.index[end])\n\n    return pd.Series(values, index=dates, name=f"HS_VaR_{alpha:g}")\n\n\ndef forecast_variance_covariance_var_rolling(\n    returns: pd.DataFrame,\n    alpha: float = 0.05,\n    weights: Iterable[float] = (0.5, 0.5),\n    n_insample: int = 1135,\n    n_oos: int = 500,\n    include_mean: bool = True,\n) -> pd.Series:\n    """Rolling variance-covariance VaR under conditional normality."""\n    w = validate_alpha_weights(alpha, weights)\n    frame = prepare_bivariate_returns(returns, n_insample, n_oos)\n    z_alpha = norm.ppf(alpha)\n    values, dates = [], []\n\n    for _, date, window in rolling_windows(frame, n_insample, n_oos):\n        mu = window.mean().to_numpy(dtype=float) if include_mean else np.zeros(2)\n        cov = window.cov().to_numpy(dtype=float)\n        port_mu = float(w @ mu)\n        port_var = float(w @ cov @ w)\n        values.append(port_mu + np.sqrt(max(port_var, 0.0)) * z_alpha)\n        dates.append(date)\n\n    return pd.Series(values, index=dates, name=f"VarCov_VaR_{alpha:g}")\n\n\ndef forecast_riskmetrics_var_rolling(\n    returns: pd.DataFrame,\n    alpha: float = 0.05,\n    weights: Iterable[float] = (0.5, 0.5),\n    lambda_: float = 0.94,\n    n_insample: int = 1135,\n    n_oos: int = 500,\n    include_mean: bool = False,\n) -> pd.Series:\n    """Paper-like RiskMetrics VaR using scalar EWMA variance of portfolio returns.\n\n    sigma^2_{p,t|t-1} = (1-lambda) r^2_{p,t-1} + lambda sigma^2_{p,t-1|t-2}.\n    """\n    w = validate_alpha_weights(alpha, weights)\n    if not 0.0 < lambda_ < 1.0:\n        raise ValueError("lambda_ must be in (0, 1).")\n\n    frame = prepare_bivariate_returns(returns, n_insample, n_oos)\n    portfolio = frame @ w\n    z_alpha = norm.ppf(alpha)\n\n    initial_window = portfolio.iloc[:n_insample]\n    sigma2 = float(initial_window.var(ddof=1))\n    values, dates = [], []\n\n    for i in range(n_oos):\n        forecast_pos = i + n_insample\n        # The forecast for date t uses r_{t-1}.\n        r_lag = float(portfolio.iloc[forecast_pos - 1])\n        sigma2 = (1.0 - lambda_) * r_lag**2 + lambda_ * sigma2\n        mu = float(portfolio.iloc[i:forecast_pos].mean()) if include_mean else 0.0\n        values.append(mu + np.sqrt(max(sigma2, 0.0)) * z_alpha)\n        dates.append(portfolio.index[forecast_pos])\n\n    return pd.Series(values, index=dates, name=f"RiskMetrics_VaR_{alpha:g}")\n\n\n# -----------------------------------------------------------------------------\n# Copula CDFs and conditional CDFs: P(U1 <= u1 | U2 = u2) = dC/du2\n# -----------------------------------------------------------------------------\n\n\ndef _clip_unit(x):\n    return np.clip(np.asarray(x, dtype=float), EPS, 1.0 - EPS)\n\n\ndef copula_cdf(u: np.ndarray, v: np.ndarray, copula_params: dict[str, float], copula: str) -> np.ndarray:\n    """Bivariate copula CDF C(u, v) for all paper copulas."""\n    copula = copula.lower()\n    u = _clip_unit(u)\n    v = _clip_unit(v)\n\n    if copula == "gaussian":\n        rho = float(copula_params["rho"])\n        z = np.column_stack([norm.ppf(u.ravel()), norm.ppf(v.ravel())])\n        corr = np.array([[1.0, rho], [rho, 1.0]])\n        out = multivariate_normal.cdf(z, mean=np.zeros(2), cov=corr)\n        return np.asarray(out).reshape(np.broadcast(u, v).shape)\n\n    if copula == "student":\n        rho = float(copula_params["rho"])\n        nu = float(copula_params["nu"])\n        x = np.column_stack([t.ppf(u.ravel(), df=nu), t.ppf(v.ravel(), df=nu)])\n        shape = np.array([[1.0, rho], [rho, 1.0]])\n        out = multivariate_t.cdf(x, loc=np.zeros(2), shape=shape, df=nu)\n        return np.asarray(out).reshape(np.broadcast(u, v).shape)\n\n    if copula == "clayton":\n        theta = float(copula_params["theta"])\n        return np.maximum((u ** (-theta) + v ** (-theta) - 1.0), EPS) ** (-1.0 / theta)\n\n    if copula == "rotated_clayton":\n        theta = float(copula_params["theta"])\n        return u + v - 1.0 + copula_cdf(1.0 - u, 1.0 - v, {"theta": theta}, "clayton")\n\n    if copula == "gumbel":\n        theta = float(copula_params["theta"])\n        x = (-np.log(u)) ** theta + (-np.log(v)) ** theta\n        return np.exp(-(x ** (1.0 / theta)))\n\n    if copula == "rotated_gumbel":\n        theta = float(copula_params["theta"])\n        return u + v - 1.0 + copula_cdf(1.0 - u, 1.0 - v, {"theta": theta}, "gumbel")\n\n    if copula == "frank":\n        theta = float(copula_params["theta"])\n        if abs(theta) < 1e-8:\n            return u * v\n        a = np.expm1(-theta * u)\n        b = np.expm1(-theta * v)\n        d = np.expm1(-theta)\n        inside = 1.0 + (a * b) / d\n        return -np.log(np.maximum(inside, EPS)) / theta\n\n    if copula == "plackett":\n        theta = float(copula_params["theta"])\n        if abs(theta - 1.0) < 1e-8:\n            return u * v\n        a = 1.0 + (theta - 1.0) * (u + v)\n        disc = np.maximum(a * a - 4.0 * theta * (theta - 1.0) * u * v, EPS)\n        return (a - np.sqrt(disc)) / (2.0 * (theta - 1.0))\n\n    if copula == "sjc":\n        tau_u = float(copula_params["tau_u"])\n        tau_l = float(copula_params["tau_l"])\n        jc = _joe_clayton_cdf(u, v, tau_u=tau_u, tau_l=tau_l)\n        rotated = u + v - 1.0 + _joe_clayton_cdf(1.0 - u, 1.0 - v, tau_u=tau_l, tau_l=tau_u)\n        return 0.5 * (jc + rotated)\n\n    raise ValueError(f"Unsupported copula: {copula}")\n\n\ndef _joe_clayton_cdf(u: np.ndarray, v: np.ndarray, tau_u: float, tau_l: float) -> np.ndarray:\n    """Joe-Clayton / BB7 CDF parameterized by upper and lower tail dependence."""\n    if not (0.0 < tau_u < 1.0 and 0.0 < tau_l < 1.0):\n        raise ValueError("SJC tau_u and tau_l must be in (0, 1).")\n    kappa = 1.0 / np.log2(2.0 - tau_u)\n    gamma = -1.0 / np.log2(tau_l)\n    a = 1.0 - (1.0 - u) ** kappa\n    b = 1.0 - (1.0 - v) ** kappa\n    s = np.maximum(a ** (-gamma) + b ** (-gamma) - 1.0, 1.0 + EPS)\n    return 1.0 - (1.0 - s ** (-1.0 / gamma)) ** (1.0 / kappa)\n\n\ndef copula_conditional_cdf_u1_given_u2(\n    u1: np.ndarray,\n    u2: np.ndarray,\n    copula_params: dict[str, float],\n    copula: str = "student",\n    finite_diff_step: float = 1e-5,\n) -> np.ndarray:\n    """Compute P(U1 <= u1 | U2 = u2) = partial C(u1,u2)/partial u2."""\n    copula = copula.lower()\n    u1 = _clip_unit(u1)\n    u2 = _clip_unit(u2)\n\n    if copula == "gaussian":\n        rho = float(copula_params["rho"])\n        z1 = norm.ppf(u1)\n        z2 = norm.ppf(u2)\n        return norm.cdf((z1 - rho * z2) / np.sqrt(1.0 - rho**2))\n\n    if copula == "student":\n        rho = float(copula_params["rho"])\n        nu = float(copula_params["nu"])\n        x1 = t.ppf(u1, df=nu)\n        x2 = t.ppf(u2, df=nu)\n        cond_df = nu + 1.0\n        cond_mean = rho * x2\n        cond_scale = np.sqrt(((nu + x2**2) * (1.0 - rho**2)) / (nu + 1.0))\n        return t.cdf((x1 - cond_mean) / cond_scale, df=cond_df)\n\n    # Numerical derivative for remaining copulas. Use central differences away\n    # from boundaries and one-sided differences near boundaries.\n    h = finite_diff_step\n    u2_low = np.maximum(EPS, u2 - h)\n    u2_high = np.minimum(1.0 - EPS, u2 + h)\n    c_high = copula_cdf(u1, u2_high, copula_params, copula)\n    c_low = copula_cdf(u1, u2_low, copula_params, copula)\n    deriv = (c_high - c_low) / (u2_high - u2_low)\n    return np.clip(deriv, EPS, 1.0 - EPS)\n\n\n# -----------------------------------------------------------------------------\n# Marginal CDF / quantile helpers\n# -----------------------------------------------------------------------------\n\n\ndef msm_conditional_cdf(y: np.ndarray | float, state_probs: np.ndarray, sigma: float, h: np.ndarray) -> np.ndarray:\n    """MSM conditional CDF for centered returns."""\n    y_values = np.asarray(y, dtype=float).reshape(-1)\n    p = _normalize_probabilities(state_probs)\n    h = _validate_positive_vector(h, "h")\n    if sigma <= 0:\n        raise ValueError("sigma must be positive.")\n    cdf = np.sum(p[:, None] * norm.cdf(y_values[None, :] / (sigma * h[:, None])), axis=0)\n    return np.clip(cdf, EPS, 1.0 - EPS)\n\n\ndef msm_conditional_quantile(\n    u: np.ndarray | float,\n    state_probs: np.ndarray,\n    sigma: float,\n    h: np.ndarray,\n    grid_size: int = 20001,\n    tail_std_multiplier: float = 10.0,\n) -> np.ndarray:\n    """Invert MSM mixture-normal CDF by monotone interpolation."""\n    uniforms = _clip_unit(u).reshape(-1)\n    p = _normalize_probabilities(state_probs)\n    h = _validate_positive_vector(h, "h")\n    if sigma <= 0:\n        raise ValueError("sigma must be positive.")\n    max_scale = sigma * float(np.max(h))\n    grid = np.linspace(-tail_std_multiplier * max_scale, tail_std_multiplier * max_scale, grid_size)\n    cdf_grid = np.sum(p[:, None] * norm.cdf(grid[None, :] / (sigma * h[:, None])), axis=0)\n    cdf_grid = np.maximum.accumulate(np.clip(cdf_grid, EPS, 1.0 - EPS))\n    return np.interp(uniforms, cdf_grid, grid)\n\n\ndef _normalize_probabilities(state_probs: np.ndarray) -> np.ndarray:\n    p = np.asarray(state_probs, dtype=float).reshape(-1)\n    if np.any(p < 0) or not np.isfinite(p).all():\n        raise ValueError("state probabilities must be finite and non-negative.")\n    total = p.sum()\n    if total <= 0:\n        raise ValueError("state probabilities must sum to a positive value.")\n    return p / total\n\n\ndef _validate_positive_vector(x: np.ndarray, name: str) -> np.ndarray:\n    arr = np.asarray(x, dtype=float).reshape(-1)\n    if np.any(arr <= 0) or not np.isfinite(arr).all():\n        raise ValueError(f"{name} must contain finite positive values.")\n    return arr\n\n\ndef _unit_interval_gauss_legendre(n_nodes: int) -> tuple[np.ndarray, np.ndarray]:\n    if n_nodes < 51:\n        raise ValueError("n_nodes should be at least 51.")\n    x, w = np.polynomial.legendre.leggauss(n_nodes)\n    nodes = np.clip(0.5 * (x + 1.0), EPS, 1.0 - EPS)\n    weights = 0.5 * w\n    return nodes, weights\n\n\n# -----------------------------------------------------------------------------\n# Copula-based portfolio CDF and VaR solvers\n# -----------------------------------------------------------------------------\n\n\ndef portfolio_cdf_from_margins_and_copula(\n    q: float,\n    inverse_cdf_2: Callable[[np.ndarray], np.ndarray],\n    cdf_1: Callable[[np.ndarray], np.ndarray],\n    copula_params: dict[str, float],\n    copula: str,\n    pi: float = 0.5,\n    integration_nodes: int = 501,\n) -> float:\n    """Generic copula portfolio CDF for two continuous conditional margins."""\n    if not 0.0 < pi < 1.0:\n        raise ValueError("pi must be in (0, 1).")\n    nodes, weights = _unit_interval_gauss_legendre(integration_nodes)\n    r2 = inverse_cdf_2(nodes)\n    r1_threshold = (q - (1.0 - pi) * r2) / pi\n    u1_threshold = cdf_1(r1_threshold)\n    cond = copula_conditional_cdf_u1_given_u2(u1_threshold, nodes, copula_params, copula)\n    return float(np.sum(weights * cond))\n\n\ndef solve_portfolio_var(\n    alpha: float,\n    inverse_cdf_1: Callable[[np.ndarray], np.ndarray],\n    inverse_cdf_2: Callable[[np.ndarray], np.ndarray],\n    cdf_1: Callable[[np.ndarray], np.ndarray],\n    copula_params: dict[str, float],\n    copula: str,\n    pi: float = 0.5,\n    integration_nodes: int = 501,\n    root_tol: float = 1e-4,\n) -> float:\n    """Solve F_p(q) = alpha for the signed portfolio return VaR."""\n    if not 0.0 < alpha < 1.0:\n        raise ValueError("alpha must be in (0, 1).")\n    if pi == 1.0:\n        return float(inverse_cdf_1(np.array([alpha]))[0])\n    if pi == 0.0:\n        return float(inverse_cdf_2(np.array([alpha]))[0])\n\n    # Conservative bracket based on near-endpoint marginal quantiles.\n    low_u, high_u = EPS, 1.0 - EPS\n    lower = pi * inverse_cdf_1(np.array([low_u]))[0] + (1.0 - pi) * inverse_cdf_2(np.array([low_u]))[0]\n    upper = pi * inverse_cdf_1(np.array([high_u]))[0] + (1.0 - pi) * inverse_cdf_2(np.array([high_u]))[0]\n\n    def objective(q: float) -> float:\n        return portfolio_cdf_from_margins_and_copula(\n            q=q,\n            inverse_cdf_2=inverse_cdf_2,\n            cdf_1=cdf_1,\n            copula_params=copula_params,\n            copula=copula,\n            pi=pi,\n            integration_nodes=integration_nodes,\n        ) - alpha\n\n    f_low = objective(lower)\n    f_high = objective(upper)\n    expand = 0\n    while not (f_low <= 0.0 <= f_high):\n        width = upper - lower\n        lower -= width\n        upper += width\n        f_low = objective(lower)\n        f_high = objective(upper)\n        expand += 1\n        if expand > 12:\n            raise RuntimeError(f"Unable to bracket VaR root: [{lower}, {upper}], f=[{f_low}, {f_high}]")\n\n    return float(brentq(objective, lower, upper, xtol=root_tol, rtol=1e-6, maxiter=100))\n\n\n# -----------------------------------------------------------------------------\n# Rolling Copula-MSM VaR\n# -----------------------------------------------------------------------------\n\n\ndef forecast_msm_copula_var_rolling(\n    returns: pd.DataFrame,\n    copula: str = "student",\n    alpha: float = 0.05,\n    weights: Iterable[float] = (0.5, 0.5),\n    k: int = 5,\n    n_insample: int = 1135,\n    n_oos: int = 500,\n    n_starts: int = 10,\n    seed: int = 123,\n    integration_nodes: int = 501,\n    root_tol: float = 1e-4,\n    verbose: bool = True,\n) -> pd.Series:\n    """Rolling one-step-ahead Copula-MSM VaR. Paper default uses k=5."""\n    try:\n        from src.msm import (\n            fit_msm,\n            make_msm_states,\n            msm_filter_from_result,\n            renewal_probabilities_from_gamma_k,\n            transition_matrix_from_gammas,\n        )\n        from src.copulas import fit_copula\n    except ImportError:\n        from msm import (\n            fit_msm,\n            make_msm_states,\n            msm_filter_from_result,\n            renewal_probabilities_from_gamma_k,\n            transition_matrix_from_gammas,\n        )\n        from copulas import fit_copula\n\n    w = validate_alpha_weights(alpha, weights)\n    pi = float(w[0])\n    frame = prepare_bivariate_returns(returns, n_insample, n_oos)\n    assets = list(frame.columns)\n    values, dates = [], []\n\n    for i, date, window in rolling_windows(frame, n_insample, n_oos):\n        r1, r2 = window[assets[0]], window[assets[1]]\n        msm_1 = fit_msm(r1, k=k, n_starts=n_starts, seed=seed + 10000 * i + 1, verbose=False)\n        msm_2 = fit_msm(r2, k=k, n_starts=n_starts, seed=seed + 10000 * i + 2, verbose=False)\n\n        filt_1 = msm_filter_from_result(r1, msm_1)\n        filt_2 = msm_filter_from_result(r2, msm_2)\n        pit = pd.concat([filt_1["pit"].rename(assets[0]), filt_2["pit"].rename(assets[1])], axis=1).dropna()\n        cop_fit = fit_copula(pit, copula=copula, margin_model="MSM")\n\n        gammas_1 = renewal_probabilities_from_gamma_k(k=k, b=msm_1.params.b, gamma_k=msm_1.params.gamma_k)\n        gammas_2 = renewal_probabilities_from_gamma_k(k=k, b=msm_2.params.b, gamma_k=msm_2.params.gamma_k)\n        A_1 = transition_matrix_from_gammas(gammas_1)\n        A_2 = transition_matrix_from_gammas(gammas_2)\n        p1 = filt_1["filtered_probs"].iloc[-1].to_numpy(dtype=float) @ A_1\n        p2 = filt_2["filtered_probs"].iloc[-1].to_numpy(dtype=float) @ A_2\n\n        h1 = np.sqrt(np.prod(make_msm_states(k=k, m0=msm_1.params.m0), axis=1))\n        h2 = np.sqrt(np.prod(make_msm_states(k=k, m0=msm_2.params.m0), axis=1))\n\n        def inv1(u):\n            return msm_1.mean_return + msm_conditional_quantile(u, p1, msm_1.params.sigma, h1)\n\n        def inv2(u):\n            return msm_2.mean_return + msm_conditional_quantile(u, p2, msm_2.params.sigma, h2)\n\n        def cdf1(x):\n            return msm_conditional_cdf(np.asarray(x) - msm_1.mean_return, p1, msm_1.params.sigma, h1)\n\n        var_t = solve_portfolio_var(\n            alpha=alpha,\n            inverse_cdf_1=inv1,\n            inverse_cdf_2=inv2,\n            cdf_1=cdf1,\n            copula_params=cop_fit.params,\n            copula=copula,\n            pi=pi,\n            integration_nodes=integration_nodes,\n            root_tol=root_tol,\n        )\n        values.append(var_t)\n        dates.append(date)\n\n        if verbose and (i + 1) % 10 == 0:\n            print(f"MSM-{copula} alpha={alpha:g}: {i + 1}/{n_oos}, VaR={var_t:.4f}")\n\n    return pd.Series(values, index=dates, name=f"CopulaMSM_{copula}_VaR_{alpha:g}")\n\n\n# -----------------------------------------------------------------------------\n# Rolling GARCH, Copula-GARCH, CCC-GARCH\n# -----------------------------------------------------------------------------\n\n\ndef _fit_arch_garch_11(series: pd.Series, dist: str = "normal"):\n    try:\n        from arch import arch_model\n    except ImportError as exc:\n        raise ImportError("Install the `arch` package: pip install arch") from exc\n    model = arch_model(series, mean="Constant", vol="GARCH", p=1, q=1, dist=dist, rescale=False)\n    return model.fit(disp="off")\n\n\ndef _garch_one_step_forecast(result) -> tuple[float, float]:\n    """Return one-step-ahead mean and volatility from an arch result."""\n    params = result.params\n    mu = float(params.get("mu", 0.0))\n    forecast = result.forecast(horizon=1, reindex=False)\n    variance = float(forecast.variance.iloc[-1, 0])\n    return mu, np.sqrt(max(variance, 0.0))\n\n\ndef forecast_ccc_garch_var_rolling(\n    returns: pd.DataFrame,\n    alpha: float = 0.05,\n    weights: Iterable[float] = (0.5, 0.5),\n    n_insample: int = 1135,\n    n_oos: int = 500,\n    include_mean: bool = True,\n    verbose: bool = True,\n) -> pd.Series:\n    """Rolling CCC-GARCH VaR with univariate GARCH(1,1)-normal margins."""\n    w = validate_alpha_weights(alpha, weights)\n    frame = prepare_bivariate_returns(returns, n_insample, n_oos)\n    assets = list(frame.columns)\n    z_alpha = norm.ppf(alpha)\n    values, dates = [], []\n\n    for i, date, window in rolling_windows(frame, n_insample, n_oos):\n        fits = [_fit_arch_garch_11(window[a]) for a in assets]\n        forecasts = [_garch_one_step_forecast(fit) for fit in fits]\n        mus = np.array([m for m, _ in forecasts], dtype=float)\n        sigmas = np.array([s for _, s in forecasts], dtype=float)\n\n        std_resids = pd.concat(\n            [fit.std_resid.dropna().rename(asset) for fit, asset in zip(fits, assets)], axis=1\n        ).dropna()\n        rho = float(std_resids.corr().iloc[0, 1])\n        corr = np.array([[1.0, rho], [rho, 1.0]])\n        D = np.diag(sigmas)\n        cov = D @ corr @ D\n        port_mu = float(w @ mus) if include_mean else 0.0\n        port_var = float(w @ cov @ w)\n        var_t = port_mu + np.sqrt(max(port_var, 0.0)) * z_alpha\n        values.append(var_t)\n        dates.append(date)\n\n        if verbose and (i + 1) % 25 == 0:\n            print(f"CCC-GARCH alpha={alpha:g}: {i + 1}/{n_oos}, VaR={var_t:.4f}")\n\n    return pd.Series(values, index=dates, name=f"CCC_GARCH_VaR_{alpha:g}")\n\n\ndef forecast_garch_copula_var_rolling(\n    returns: pd.DataFrame,\n    copula: str = "student",\n    alpha: float = 0.05,\n    weights: Iterable[float] = (0.5, 0.5),\n    n_insample: int = 1135,\n    n_oos: int = 500,\n    integration_nodes: int = 501,\n    root_tol: float = 1e-4,\n    verbose: bool = True,\n) -> pd.Series:\n    """Rolling one-step-ahead Copula-GARCH VaR with Gaussian GARCH margins."""\n    try:\n        from src.copulas import fit_copula\n    except ImportError:\n        from copulas import fit_copula\n\n    w = validate_alpha_weights(alpha, weights)\n    pi = float(w[0])\n    frame = prepare_bivariate_returns(returns, n_insample, n_oos)\n    assets = list(frame.columns)\n    values, dates = [], []\n\n    for i, date, window in rolling_windows(frame, n_insample, n_oos):\n        fits = [_fit_arch_garch_11(window[a]) for a in assets]\n        forecasts = [_garch_one_step_forecast(fit) for fit in fits]\n        mu1, sigma1 = forecasts[0]\n        mu2, sigma2 = forecasts[1]\n\n        pits = []\n        for fit, asset in zip(fits, assets):\n            std = fit.std_resid.dropna()\n            pits.append(pd.Series(norm.cdf(std), index=std.index, name=asset).clip(EPS, 1.0 - EPS))\n        pit = pd.concat(pits, axis=1).dropna()\n        cop_fit = fit_copula(pit, copula=copula, margin_model="GARCH")\n\n        def inv1(u):\n            return mu1 + sigma1 * norm.ppf(_clip_unit(u))\n\n        def inv2(u):\n            return mu2 + sigma2 * norm.ppf(_clip_unit(u))\n\n        def cdf1(x):\n            return norm.cdf((np.asarray(x, dtype=float) - mu1) / sigma1)\n\n        var_t = solve_portfolio_var(\n            alpha=alpha,\n            inverse_cdf_1=inv1,\n            inverse_cdf_2=inv2,\n            cdf_1=cdf1,\n            copula_params=cop_fit.params,\n            copula=copula,\n            pi=pi,\n            integration_nodes=integration_nodes,\n            root_tol=root_tol,\n        )\n        values.append(var_t)\n        dates.append(date)\n\n        if verbose and (i + 1) % 25 == 0:\n            print(f"GARCH-{copula} alpha={alpha:g}: {i + 1}/{n_oos}, VaR={var_t:.4f}")\n\n    return pd.Series(values, index=dates, name=f"CopulaGARCH_{copula}_VaR_{alpha:g}")\n\n\n# -----------------------------------------------------------------------------\n# Convenience wrappers and backtest inputs\n# -----------------------------------------------------------------------------\n\n\ndef portfolio_returns(returns: pd.DataFrame, weights: Iterable[float] = (0.5, 0.5)) -> pd.Series:\n    w = validate_alpha_weights(0.05, weights)\n    frame = returns.apply(pd.to_numeric, errors="coerce").dropna(how="any")\n    out = frame @ w\n    out.name = "portfolio_return"\n    return out\n\n\ndef var_exceedances(realized_portfolio_returns: pd.Series, var_forecasts: pd.Series) -> pd.Series:\n    """VaR hit sequence I_t = 1{r_p,t < VaR_t}."""\n    aligned = pd.concat([realized_portfolio_returns.rename("r"), var_forecasts.rename("VaR")], axis=1).dropna()\n    return aligned["r"].lt(aligned["VaR"]).astype(int).rename("hit")\n\n\ndef violation_frequency(realized_portfolio_returns: pd.Series, var_forecasts: pd.Series) -> float:\n    return float(var_exceedances(realized_portfolio_returns, var_forecasts).mean())\n\n\ndef forecast_all_var_models(\n    returns: pd.DataFrame,\n    alpha: float = 0.05,\n    weights: Iterable[float] = (0.5, 0.5),\n    n_insample: int = 1135,\n    n_oos: int = 500,\n    copulas: Iterable[str] = SUPPORTED_COPULAS,\n    include_msm: bool = True,\n    include_garch_copula: bool = True,\n    msm_k: int = 5,\n    msm_n_starts: int = 10,\n    integration_nodes: int = 501,\n    verbose: bool = True,\n) -> pd.DataFrame:\n    """Compute the full paper-like VaR panel for one alpha.\n\n    This is computationally expensive, especially Copula-MSM rolling.\n    """\n    results: dict[str, pd.Series] = {}\n\n    results["Historical"] = forecast_historical_var_rolling(returns, alpha, weights, n_insample, n_oos)\n    results["RiskMetrics"] = forecast_riskmetrics_var_rolling(returns, alpha, weights, 0.94, n_insample, n_oos)\n    results["Var-Covar"] = forecast_variance_covariance_var_rolling(returns, alpha, weights, n_insample, n_oos)\n    results["CCC-GARCH"] = forecast_ccc_garch_var_rolling(returns, alpha, weights, n_insample, n_oos, verbose=verbose)\n\n    if include_garch_copula:\n        for copula in copulas:\n            results[f"Copula-GARCH {copula}"] = forecast_garch_copula_var_rolling(\n                returns=returns,\n                copula=copula,\n                alpha=alpha,\n                weights=weights,\n                n_insample=n_insample,\n                n_oos=n_oos,\n                integration_nodes=integration_nodes,\n                verbose=verbose,\n            )\n\n    if include_msm:\n        for copula in copulas:\n            results[f"Copula-MSM {copula}"] = forecast_msm_copula_var_rolling(\n                returns=returns,\n                copula=copula,\n                alpha=alpha,\n                weights=weights,\n                k=msm_k,\n                n_insample=n_insample,\n                n_oos=n_oos,\n                n_starts=msm_n_starts,\n                integration_nodes=integration_nodes,\n                verbose=verbose,\n            )\n\n    return pd.concat(results, axis=1)\n'

(SRC_DIR / 'msm.py').write_text(MSM_CODE, encoding='utf-8')
(SRC_DIR / 'copulas.py').write_text(COPULAS_CODE, encoding='utf-8')
(PROJECT_DIR / 'var_v2.py').write_text(VAR_V2_CODE, encoding='utf-8')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

print('Modules written to:', PROJECT_DIR)
print('Files:', sorted([p.name for p in SRC_DIR.iterdir()]))


## 4. Imports

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from var_v2 import (
    SUPPORTED_COPULAS,
    prepare_bivariate_returns,
    forecast_historical_var_rolling,
    forecast_variance_covariance_var_rolling,
    forecast_riskmetrics_var_rolling,
    forecast_ccc_garch_var_rolling,
    forecast_garch_copula_var_rolling,
    forecast_msm_copula_var_rolling,
    portfolio_returns,
    var_exceedances,
    violation_frequency,
)

print('Supported copulas:', SUPPORTED_COPULAS)

## 5. Chemins Drive et chargement du CSV des rendements

Place ton fichier `returns_nasdaq_sp500.csv` dans Google Drive, par exemple directement dans `MyDrive/`.

Le CSV doit contenir deux colonnes de rendements, en pourcentage, par exemple `NASDAQ` et `SP500`, avec une colonne date ou un index date.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive')

# À adapter si ton fichier est dans un sous-dossier Drive.
RETURNS_CSV = DRIVE_ROOT / 'returns_nasdaq_sp500.csv'

OUTPUT_DIR = DRIVE_ROOT / 'copula_msm_var_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Returns CSV:', RETURNS_CSV)
print('Output directory:', OUTPUT_DIR)

In [ ]:
def load_returns_csv(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)

    # Cas fréquent : index sauvegardé comme première colonne unnamed.
    first_col = str(frame.columns[0]).lower()
    if first_col.startswith('unnamed') or 'date' in first_col or 'time' in first_col:
        frame = frame.rename(columns={frame.columns[0]: 'date'})
        frame['date'] = pd.to_datetime(frame['date'])
        frame = frame.set_index('date')
    else:
        # Si aucune colonne date évidente, on tente de parser l'index actuel plus tard.
        pass

    frame = frame.apply(pd.to_numeric, errors='coerce').dropna(how='any')
    frame = frame.sort_index()

    if frame.shape[1] != 2:
        raise ValueError(f'Expected exactly 2 return columns, got {frame.shape[1]}: {frame.columns.tolist()}')

    return frame

returns = load_returns_csv(RETURNS_CSV)

print(returns.shape)
print(returns.index.min(), '→', returns.index.max())
display(returns.head())
display(returns.describe().T)

## 6. Paramètres communs

In [ ]:
PI = 0.5
WEIGHTS = np.array([PI, 1.0 - PI])

N_OOS = 500
WINDOW_SIZE = 1135

ALPHA_5 = 0.05
ALPHA_1 = 0.01

returns_var = prepare_bivariate_returns(
    returns,
    n_insample=WINDOW_SIZE,
    n_oos=N_OOS,
)

portfolio_ret = portfolio_returns(returns_var, weights=WEIGHTS)
oos_index = returns_var.index[WINDOW_SIZE:WINDOW_SIZE + N_OOS]
portfolio_ret_oos = portfolio_ret.loc[oos_index]

print('returns_var:', returns_var.shape)
print('OOS:', len(portfolio_ret_oos), portfolio_ret_oos.index.min(), '→', portfolio_ret_oos.index.max())
assert len(portfolio_ret_oos) == N_OOS

## 7. Fonctions utilitaires de sauvegarde/reprise

In [ ]:
def save_var(series: pd.Series, filename: str) -> Path:
    path = OUTPUT_DIR / filename
    series.to_frame(name=series.name or 'VaR').to_csv(path)
    print('Saved:', path)
    return path


def load_var(filename: str, name: str | None = None) -> pd.Series:
    path = OUTPUT_DIR / filename
    frame = pd.read_csv(path, index_col=0, parse_dates=True)
    s = frame.iloc[:, 0]
    if name is not None:
        s.name = name
    return s


def run_or_load(filename: str, name: str, compute_fn, force: bool = False) -> pd.Series:
    path = OUTPUT_DIR / filename
    if path.exists() and not force:
        print('Loading existing file:', path)
        return load_var(filename, name=name)
    print('Computing:', name)
    series = compute_fn().rename(name)
    save_var(series, filename)
    return series


def concat_saved_vars(file_map: dict[str, str]) -> pd.DataFrame:
    series = {name: load_var(filename, name=name) for name, filename in file_map.items()}
    panel = pd.concat(series, axis=1).dropna(how='any')
    print('Panel shape:', panel.shape)
    return panel


def simple_hit_summary(var_panel: pd.DataFrame, alpha: float) -> pd.DataFrame:
    rows = []
    for col in var_panel.columns:
        hits = var_exceedances(portfolio_ret_oos, var_panel[col])
        rows.append({
            'model': col,
            'alpha': alpha,
            'violations': int(hits.sum()),
            'EFV': float(hits.mean()),
            'expected_violations': alpha * len(hits),
            'nobs': len(hits),
        })
    return pd.DataFrame(rows).set_index('model')

# 8. Lancer les VaR une par une

Chaque cellule calcule un modèle et sauvegarde immédiatement le CSV dans `OUTPUT_DIR` sur Drive.

Pour relancer un modèle déjà sauvegardé, mets `force=True` dans `run_or_load(...)`.

## 8.1 VaR 5% — benchmarks rapides

In [ ]:
var_hist_5 = run_or_load(
    filename='var_hist_5.csv',
    name='Historical',
    compute_fn=lambda: forecast_historical_var_rolling(
        returns_var, alpha=ALPHA_5, weights=WEIGHTS,
        n_insample=WINDOW_SIZE, n_oos=N_OOS,
    ),
    force=False,
)
var_hist_5.head()

In [ ]:
var_vc_5 = run_or_load(
    filename='var_vc_5.csv',
    name='Variance-Covariance',
    compute_fn=lambda: forecast_variance_covariance_var_rolling(
        returns_var, alpha=ALPHA_5, weights=WEIGHTS,
        n_insample=WINDOW_SIZE, n_oos=N_OOS, include_mean=True,
    ),
    force=False,
)
var_vc_5.head()

In [ ]:
var_rm_5 = run_or_load(
    filename='var_rm_5.csv',
    name='RiskMetrics',
    compute_fn=lambda: forecast_riskmetrics_var_rolling(
        returns_var, alpha=ALPHA_5, weights=WEIGHTS,
        lambda_=0.94, n_insample=WINDOW_SIZE, n_oos=N_OOS,
        include_mean=False,
    ),
    force=False,
)
var_rm_5.head()

In [ ]:
var_ccc_5 = run_or_load(
    filename='var_ccc_5.csv',
    name='CCC-GARCH',
    compute_fn=lambda: forecast_ccc_garch_var_rolling(
        returns_var, alpha=ALPHA_5, weights=WEIGHTS,
        n_insample=WINDOW_SIZE, n_oos=N_OOS,
        include_mean=True, verbose=True,
    ),
    force=False,
)
var_ccc_5.head()

## 8.2 VaR 5% — Copula-GARCH

Lance une copule à la fois en changeant `COPULA_TO_RUN`.

In [ ]:
COPULA_TO_RUN = 'student'  # gaussian, student, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_garch_copula_5 = run_or_load(
    filename=f'var_garch_{COPULA_TO_RUN}_5.csv',
    name=f'Copula-GARCH {COPULA_TO_RUN}',
    compute_fn=lambda: forecast_garch_copula_var_rolling(
        returns_var,
        copula=COPULA_TO_RUN,
        alpha=ALPHA_5,
        weights=WEIGHTS,
        n_insample=WINDOW_SIZE,
        n_oos=N_OOS,
        integration_nodes=501,
        root_tol=1e-4,
        verbose=True,
    ),
    force=False,
)
var_garch_copula_5.head()

## 8.3 VaR 5% — Copula-MSM

C'est la partie la plus lente. Pour la réplication papier, garder `k=5`. Pour un test rapide, réduire temporairement `N_OOS`, `n_starts` et `integration_nodes` dans l'appel ci-dessous.

In [ ]:
COPULA_TO_RUN = 'student'  # gaussian, student, plackett, clayton, rotated_clayton, sjc, frank, gumbel, rotated_gumbel

var_msm_copula_5 = run_or_load(
    filename=f'var_msm_{COPULA_TO_RUN}_5.csv',
    name=f'Copula-MSM {COPULA_TO_RUN}',
    compute_fn=lambda: forecast_msm_copula_var_rolling(
        returns_var,
        copula=COPULA_TO_RUN,
        alpha=ALPHA_5,
        weights=WEIGHTS,
        k=5,
        n_insample=WINDOW_SIZE,
        n_oos=N_OOS,
        n_starts=10,
        seed=123,
        integration_nodes=501,
        root_tol=1e-4,
        verbose=True,
    ),
    force=False,
)
var_msm_copula_5.head()

# 9. VaR 1%

Même logique pour `alpha=1%`. Tu peux faire tourner seulement les modèles/couples nécessaires sur Colab et laisser le reste à VSCode.

## 9.1 VaR 1% — benchmarks rapides

In [ ]:
var_hist_1 = run_or_load(
    filename='var_hist_1.csv',
    name='Historical',
    compute_fn=lambda: forecast_historical_var_rolling(
        returns_var, alpha=ALPHA_1, weights=WEIGHTS,
        n_insample=WINDOW_SIZE, n_oos=N_OOS,
    ),
    force=False,
)
var_hist_1.head()

In [ ]:
var_vc_1 = run_or_load(
    filename='var_vc_1.csv',
    name='Variance-Covariance',
    compute_fn=lambda: forecast_variance_covariance_var_rolling(
        returns_var, alpha=ALPHA_1, weights=WEIGHTS,
        n_insample=WINDOW_SIZE, n_oos=N_OOS, include_mean=True,
    ),
    force=False,
)
var_vc_1.head()

In [ ]:
var_rm_1 = run_or_load(
    filename='var_rm_1.csv',
    name='RiskMetrics',
    compute_fn=lambda: forecast_riskmetrics_var_rolling(
        returns_var, alpha=ALPHA_1, weights=WEIGHTS,
        lambda_=0.94, n_insample=WINDOW_SIZE, n_oos=N_OOS,
        include_mean=False,
    ),
    force=False,
)
var_rm_1.head()

In [ ]:
var_ccc_1 = run_or_load(
    filename='var_ccc_1.csv',
    name='CCC-GARCH',
    compute_fn=lambda: forecast_ccc_garch_var_rolling(
        returns_var, alpha=ALPHA_1, weights=WEIGHTS,
        n_insample=WINDOW_SIZE, n_oos=N_OOS,
        include_mean=True, verbose=True,
    ),
    force=False,
)
var_ccc_1.head()

## 9.2 VaR 1% — Copula-GARCH

In [ ]:
COPULA_TO_RUN = 'student'

var_garch_copula_1 = run_or_load(
    filename=f'var_garch_{COPULA_TO_RUN}_1.csv',
    name=f'Copula-GARCH {COPULA_TO_RUN}',
    compute_fn=lambda: forecast_garch_copula_var_rolling(
        returns_var,
        copula=COPULA_TO_RUN,
        alpha=ALPHA_1,
        weights=WEIGHTS,
        n_insample=WINDOW_SIZE,
        n_oos=N_OOS,
        integration_nodes=501,
        root_tol=1e-4,
        verbose=True,
    ),
    force=False,
)
var_garch_copula_1.head()

## 9.3 VaR 1% — Copula-MSM

In [ ]:
COPULA_TO_RUN = 'student'

var_msm_copula_1 = run_or_load(
    filename=f'var_msm_{COPULA_TO_RUN}_1.csv',
    name=f'Copula-MSM {COPULA_TO_RUN}',
    compute_fn=lambda: forecast_msm_copula_var_rolling(
        returns_var,
        copula=COPULA_TO_RUN,
        alpha=ALPHA_1,
        weights=WEIGHTS,
        k=5,
        n_insample=WINDOW_SIZE,
        n_oos=N_OOS,
        n_starts=10,
        seed=123,
        integration_nodes=501,
        root_tol=1e-4,
        verbose=True,
    ),
    force=False,
)
var_msm_copula_1.head()

# 10. Concaténer les VaR sauvegardées

In [ ]:
# Modifie ce dictionnaire selon les fichiers réellement calculés.
file_map_5 = {
    'Historical': 'var_hist_5.csv',
    'Variance-Covariance': 'var_vc_5.csv',
    'RiskMetrics': 'var_rm_5.csv',
    'CCC-GARCH': 'var_ccc_5.csv',
    'Copula-GARCH student': 'var_garch_student_5.csv',
    'Copula-MSM student': 'var_msm_student_5.csv',
}

# On retire automatiquement les fichiers absents.
file_map_5 = {name: fn for name, fn in file_map_5.items() if (OUTPUT_DIR / fn).exists()}

var_panel_5 = concat_saved_vars(file_map_5)
var_panel_5.to_csv(OUTPUT_DIR / 'var_panel_5.csv')
display(var_panel_5.head())
display(simple_hit_summary(var_panel_5, alpha=ALPHA_5))

In [ ]:
# Modifie ce dictionnaire selon les fichiers réellement calculés.
file_map_1 = {
    'Historical': 'var_hist_1.csv',
    'Variance-Covariance': 'var_vc_1.csv',
    'RiskMetrics': 'var_rm_1.csv',
    'CCC-GARCH': 'var_ccc_1.csv',
    'Copula-GARCH student': 'var_garch_student_1.csv',
    'Copula-MSM student': 'var_msm_student_1.csv',
}

file_map_1 = {name: fn for name, fn in file_map_1.items() if (OUTPUT_DIR / fn).exists()}

var_panel_1 = concat_saved_vars(file_map_1)
var_panel_1.to_csv(OUTPUT_DIR / 'var_panel_1.csv')
display(var_panel_1.head())
display(simple_hit_summary(var_panel_1, alpha=ALPHA_1))

# 11. Debug rapide sur 20 jours

À utiliser avant un run long : vérifie que le modèle tourne sans erreur.

In [ ]:
# Exemple debug rapide Copula-MSM Student 5% sur 20 jours.
# Décommente pour tester.

# var_debug = forecast_msm_copula_var_rolling(
#     returns_var,
#     copula='student',
#     alpha=ALPHA_5,
#     weights=WEIGHTS,
#     k=5,
#     n_insample=WINDOW_SIZE,
#     n_oos=20,
#     n_starts=3,
#     seed=123,
#     integration_nodes=101,
#     root_tol=1e-4,
#     verbose=True,
# )
# var_debug